In [1]:
import pandas as pd
import re

# 1. Đọc dữ liệu từ file csv
df = pd.read_csv('/kaggle/input/datasets/mingchan1/dataset/data_cleaned_final.csv')

# Định nghĩa các pattern (biểu thức chính quy) để tìm kiếm các định dạng thời gian
date_patterns = [
    r'\b\d{1,2}\s+tháng\s+\d{1,2}\s+năm\s+\d{4}\b',  # VD: 18 tháng 11 năm 1834
    r'\b\d{1,2}\s+tháng\s+\d{1,2}\b',                # VD: 18 tháng 11
    r'\b\d{1,2}[\.\/\-]\d{1,2}[\.\/\-]\d{2,4}\b',    # VD: 3.4.1946, 11/07/1992
    r'\bnăm\s+\d{4}\b',                              # VD: năm 1968
    r'(?<!\d)\b(1[0-9]|20)\d{2}\b(?!\s*(người|km|kg|m|KB|MB|GB))' # VD: 1968, 1834 (các năm đứng độc lập)
]
combined_pattern = '|'.join(date_patterns)

# Hàm thay thế thời gian thành [MASK]
def mask_time(text):
    if pd.isna(text):
        return text
    # Tìm và thay thế tất cả thời gian khớp với pattern bằng chuỗi [MASK]
    return re.sub(combined_pattern, '[MASK]', str(text), flags=re.IGNORECASE)

# Thực hiện Yêu cầu 1: Làm mờ dữ liệu thời gian trong cột sự kiện
# (Tôi tạo cột mới 'Sự kiện_Masked' để bạn có thể so sánh với cột cũ)
df['Sự kiện_Masked'] = df['Sự kiện'].apply(mask_time)

# Thực hiện Yêu cầu 2: Đánh số thứ tự cho từng câu, tạo thành dãy sự kiện từ 1 đến hết theo từng người
df['Thứ tự'] = df.groupby('Nhân vật').cumcount() + 1

# Sắp xếp lại thứ tự các cột
cols = ['Nhân vật', 'Thứ tự', 'Năm', 'Sự kiện', 'Sự kiện_Masked']
df_out = df[cols] if all(c in df.columns for c in cols) else df

# Xuất ra file csv mới
df_out.to_csv('data_processed.csv', index=False)

print("Đã xử lý xong!")

Đã xử lý xong!


In [2]:
# CELL 1: CÀI ĐẶT VÀ IMPORT THƯ VIỆN (BẢN AN TOÀN CHO KAGGLE)

# Chỉ cài thêm transformers nếu thiếu, không phá vỡ môi trường Kaggle gốc
!pip install -q transformers tqdm

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Bỏ AdamW ra khỏi transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Import AdamW từ thư viện gốc của PyTorch
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import random

# Cố định random seed để kết quả ổn định qua nhiều lần chạy
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")
df = pd.read_csv('/kaggle/working/data_processed.csv')
df = df.sort_values(by=['Nhân vật', 'Thứ tự'])

# 1. Tách 80% Nhân vật cho Train, 20% cho Test
all_characters = df['Nhân vật'].unique()
train_chars, test_chars = train_test_split(all_characters, test_size=0.2, random_state=42)

df_train = df[df['Nhân vật'].isin(train_chars)]
df_test = df[df['Nhân vật'].isin(test_chars)]

print(f"Tổng số nhân vật: {len(all_characters)} | Train: {len(train_chars)} | Test: {len(test_chars)}")

# 2. Tạo dữ liệu Train Pairwise (Chỉ dùng tập df_train, kết hợp Window Sampling)
pairs = []
labels = []
WINDOW_SIZE = 4

for name, group in tqdm(df_train.groupby('Nhân vật'), desc="Tạo Train Data"):
    events = group['Sự kiện'].tolist()
    n = len(events)
    for i in range(n):
        end_idx = min(i + 1 + WINDOW_SIZE, n)
        for j in range(i + 1, end_idx):
            pairs.append((f"{name} - {events[i]}", events[j]))
            labels.append(1)
            pairs.append((f"{name} - {events[j]}", events[i]))
            labels.append(0)

train_df = pd.DataFrame({'text_a': [p[0] for p in pairs], 'text_b': [p[1] for p in pairs], 'label': labels})
print(f"Tạo được {len(train_df)} cặp câu cho tập Train.")

Đang sử dụng thiết bị: cuda
Tổng số nhân vật: 5841 | Train: 4672 | Test: 1169


Tạo Train Data:   0%|          | 0/4672 [00:00<?, ?it/s]

Tạo được 276376 cặp câu cho tập Train.


In [3]:
# =======================================================
# CELL ĐỘC LẬP: LOAD DATA + TIẾP TỤC / BẮT ĐẦU HUẤN LUYỆN
# Chạy được hoàn toàn độc lập, không cần cell nào chạy trước
# =======================================================

import warnings, torch, torch.nn as nn, scipy.stats as stats
import numpy as np, pandas as pd, gc, os, json
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
from transformers.utils import logging
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore")
logging.set_verbosity_error()
plt.style.use('seaborn-v0_8-whitegrid')
import matplotlib; matplotlib.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']



# --- Đường dẫn INPUT (chỉ đọc — từ Kaggle Dataset) ---
INPUT_DIR             = "/kaggle/input/datasets/mingchan1/model-input"
CHECKPOINT_FILE_READ  = f"{INPUT_DIR}/last_checkpoint.pth"
RUNTIME_STATE_READ    = f"{INPUT_DIR}/runtime_state.json"
EVAL_METRICS_READ     = f"{INPUT_DIR}/evaluation_metrics.csv"
EVAL_PREDICTIONS_READ = f"{INPUT_DIR}/evaluation_predictions.csv"

# --- Đường dẫn OUTPUT (ghi được — working dir) ---
OUTPUT_DIR            = "/kaggle/working"
CHECKPOINT_FILE       = f"{OUTPUT_DIR}/last_checkpoint.pth"
RUNTIME_STATE_FILE    = f"{OUTPUT_DIR}/runtime_state.json"
EVAL_METRICS_FILE     = f"{OUTPUT_DIR}/evaluation_metrics.csv"
EVAL_PREDICTIONS_FILE = f"{OUTPUT_DIR}/evaluation_predictions.csv"
SAVE_PATH             = f"{OUTPUT_DIR}/best_model"

# --- Cấu hình chung ---
DATA_TRAIN_PATH  = "train_data.csv"
DATA_TEST_PATH   = "test_data.csv"
FAST_MODE        = False
EVENT_THRESHOLD  = 5
EPOCHS, PATIENCE = 20, 4
os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =======================================================
# 1. LOAD DATA — ĐỘC LẬP HOÀN TOÀN
#    Ưu tiên dùng biến RAM (nếu đang chạy liên tục),
#    fallback về đọc file nếu kernel vừa reset.
# =======================================================
def _load_df(var_name, filepath, required_cols):
    """Thử lấy từ globals trước, nếu không có thì đọc file CSV."""
    import builtins

    # ★ Kiểm tra is None tường minh — tránh truth value của DataFrame
    df = globals().get(var_name, None)
    if df is None:
        df = getattr(builtins, var_name, None)

    if df is not None and isinstance(df, pd.DataFrame):
        if df.empty:
            print(f"  ⚠️  '{var_name}' có trong RAM nhưng rỗng, thử đọc file...")
        else:
            print(f"  ✔ '{var_name}' đã có trong RAM ({len(df):,} dòng)")
            return df

    # Fallback: đọc từ file
    assert os.path.exists(filepath), (
        f"Không tìm thấy '{filepath}' và '{var_name}' chưa được load vào RAM.\n"
        f"Hãy đặt file CSV tại đường dẫn trên hoặc load DataFrame trước khi chạy cell này."
    )
    df = pd.read_csv(filepath)
    missing = [c for c in required_cols if c not in df.columns]
    assert not missing, f"File '{filepath}' thiếu cột: {missing}"
    print(f"  ✔ Đọc '{filepath}' thành công ({len(df):,} dòng)")
    return df

print("📂 ĐANG LOAD DATA...")
train_df = _load_df('train_df', DATA_TRAIN_PATH, ['label', 'text_a', 'text_b'])
df_test  = _load_df('df_test',  DATA_TEST_PATH,  ['Nhân vật', 'Sự kiện_Masked'])

positive_train_df = train_df[train_df['label'] == 1].reset_index(drop=True)
if FAST_MODE:
    print("  ⚡ FAST MODE")
    train_subset = positive_train_df.sample(n=min(10000, len(positive_train_df)), random_state=42)
    VAL_MAX_ROWS = 2000
else:
    print("  🔵 FULL MODE")
    train_subset = positive_train_df
    VAL_MAX_ROWS = None
print(f"  Train: {len(train_subset):,} | Test characters: {df_test['Nhân vật'].nunique():,}\n")


# =======================================================
# 2. HÀM DÙNG CHUNG
# =======================================================
def build_prompt(char_name, event_a, event_b):
    premise    = f"Tiểu sử {char_name}. Sự kiện X: {event_a} | Sự kiện Y: {event_b}"
    hypothesis = "Sự kiện X xảy ra trước Sự kiện Y."
    return premise, hypothesis


class RankingEventDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df        = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, index):
        row   = self.df.iloc[index]
        parts = str(row['text_a']).split(" - ", 1)
        char_name = parts[0] if len(parts) == 2 else "Nhân vật"
        event_a   = parts[1] if len(parts) == 2 else str(row['text_a'])
        event_b   = str(row['text_b'])

        def enc(a, b):
            return self.tokenizer(a, b, truncation=True, max_length=self.max_len,
                                  padding='max_length', return_tensors='pt')
        pos = enc(*build_prompt(char_name, event_a, event_b))
        neg = enc(*build_prompt(char_name, event_b, event_a))
        return {
            'pos_input_ids':      pos['input_ids'].flatten(),
            'pos_attention_mask': pos['attention_mask'].flatten(),
            'neg_input_ids':      neg['input_ids'].flatten(),
            'neg_attention_mask': neg['attention_mask'].flatten(),
        }


def evaluate_timeline_batched(character_name, original_events, model,
                               tokenizer, device, batch_size=128):
    n = len(original_events)
    if n < 2:
        return 1.0, 1.0, list(range(n)), list(range(n)), [], []

    rng = np.random.default_rng(seed=42)
    shuffled_indices = list(range(n))
    rng.shuffle(shuffled_indices)
    shuffled_events = [original_events[i] for i in shuffled_indices]

    pairs, pair_indices = [], []
    for i in range(n):
        for j in range(n):
            if i == j: continue
            pairs.append(build_prompt(character_name, shuffled_events[i], shuffled_events[j]))
            pair_indices.append((i, j))

    borda_scores = np.zeros(n)
    for k in range(0, len(pairs), batch_size):
        bp = pairs[k:k + batch_size]
        inputs = tokenizer([p[0] for p in bp], [p[1] for p in bp],
                           return_tensors='pt', truncation=True,
                           max_length=128, padding='max_length')
        with torch.no_grad():
            sc = model(input_ids=inputs['input_ids'].to(device),
                       attention_mask=inputs['attention_mask'].to(device)
                       ).logits.squeeze(-1).cpu().numpy()
        for idx, s in enumerate(sc):
            i, j = pair_indices[k + idx]
            borda_scores[i] += s
        del inputs, sc
    torch.cuda.empty_cache()

    pred_order = np.argsort(borda_scores)[::-1]
    pred_mapped = [shuffled_indices[i] for i in pred_order]
    ground_truth = list(range(n))

    pred_ranks = np.zeros(n, dtype=int)
    for rank, orig in enumerate(pred_mapped):
        pred_ranks[orig] = rank + 1
    true_ranks = np.arange(1, n + 1)

    tau, _ = stats.kendalltau(ground_truth, pred_mapped)
    if np.isnan(tau):
        tau = sum(1 for a, b in zip(ground_truth, pred_mapped) if a == b) / n
    rho, _ = stats.spearmanr(true_ranks, pred_ranks)
    if np.isnan(rho): rho = 1.0

    return tau, rho, ground_truth, pred_mapped, true_ranks.tolist(), pred_ranks.tolist()

GROUP_A = (5, 10)    # inclusive
GROUP_B = (11, 15)   # inclusive
# =======================================================
# [THAY THẾ] HÀM validate_model — Thêm 2 nhóm sự kiện
# Nhóm A: 5–10 sự kiện | Nhóm B: 11–15 sự kiện
# =======================================================

# ★ SỬA 4 (đầy đủ) — thay thế hàm validate_model hoàn chỉnh
def validate_model(current_model, test_df, tokenizer, device, max_rows, epoch_num=None):
    eval_df = test_df
    if max_rows is not None:
        selected, cnt = [], 0
        for char in test_df['Nhân vật'].unique():
            selected.append(char)
            cnt += len(test_df[test_df['Nhân vật'] == char])
            if cnt >= max_rows: break
        eval_df = test_df[test_df['Nhân vật'].isin(selected)]

    scores, spr = [], []
    all_t, all_p, details = [], [], []
    sample_tl   = None
    exact_chars = 0   # ★ đếm số nhân vật khớp hoàn toàn (không phải số dòng)

    groups = {
        'A': {'range': GROUP_A, 'tau': [], 'exact': 0, 'total': 0},
        'B': {'range': GROUP_B, 'tau': [], 'exact': 0, 'total': 0},
    }

    current_model.eval()
    loop = tqdm(eval_df.groupby('Nhân vật'),
                desc=f"Validation Epoch {epoch_num}", leave=False)

    for name, group in loop:
        events = group['Sự kiện_Masked'].tolist()
        n = len(events)
        if n < 3: continue

        tau, rho, truth, pred, t_r, p_r = evaluate_timeline_batched(
            name, events, current_model, tokenizer, device)

        scores.append(tau)
        spr.append(rho)
        all_t.extend(t_r)
        all_p.extend(p_r)

        is_exact = (truth == pred)
        if is_exact:
            exact_chars += 1   # ★ đếm nhân vật, không phải dòng

        for g in groups.values():
            lo, hi = g['range']
            if lo <= n <= hi:
                g['tau'].append(tau)
                g['total'] += 1
                if is_exact:
                    g['exact'] += 1
                break

        for i in range(n):
            details.append({
                "Nhân vật": name,
                "Sự kiện":  events[i],
                "Thực tế":  t_r[i],
                "Dự đoán":  p_r[i],
                "Kết quả":  "✅" if t_r[i] == p_r[i] else "❌",
                "Nhóm":     (f"{GROUP_A[0]}–{GROUP_A[1]}" if GROUP_A[0] <= n <= GROUP_A[1]
                             else f"{GROUP_B[0]}–{GROUP_B[1]}" if GROUP_B[0] <= n <= GROUP_B[1]
                             else "khác")
            })

        if sample_tl is None and 6 <= n <= 10:
            sample_tl = (name, events, t_r, p_r)

        loop.set_postfix(Tau=f"{np.mean(scores):.4f}", Rho=f"{np.mean(spr):.4f}")

    def group_stats(g):
        return (np.mean(g['tau']) if g['tau'] else 0.0,
                g['exact'] / g['total'] * 100 if g['total'] > 0 else 0.0,
                g['total'])

    ga_tau, ga_exact, ga_n = group_stats(groups['A'])
    gb_tau, gb_exact, gb_n = group_stats(groups['B'])

    # ★ exact_rate = số nhân vật khớp hoàn toàn / tổng nhân vật × 100
    overall_exact = (exact_chars / len(scores) * 100) if scores else 0.0

    return (np.mean(scores), np.mean(spr), overall_exact,
            ga_tau, ga_exact, ga_n,
            gb_tau, gb_exact, gb_n,
            all_t, all_p, details, sample_tl)

def print_group_comparison_table(epoch, history_logs):
    """In bảng so sánh 2 nhóm qua tất cả epoch đã chạy."""
    if not history_logs:
        return

    rows = []
    for log in history_logs:
        rows.append({
            "Epoch":        int(log["Epoch"]),
            "Train Loss":   f"{log['Train_Loss']:.4f}",
            "Tau tổng":     f"{log['Val_Tau']:.4f}",
            # Nhóm A
            f"Tau {GROUP_A[0]}–{GROUP_A[1]}":   f"{log.get('GroupA_Tau',  0):.4f}",
            f"Exact {GROUP_A[0]}–{GROUP_A[1]}%": f"{log.get('GroupA_Exact', 0):.2f}%",
            f"N {GROUP_A[0]}–{GROUP_A[1]}":      int(log.get('GroupA_N', 0)),
            # Nhóm B
            f"Tau {GROUP_B[0]}–{GROUP_B[1]}":   f"{log.get('GroupB_Tau',  0):.4f}",
            f"Exact {GROUP_B[0]}–{GROUP_B[1]}%": f"{log.get('GroupB_Exact', 0):.2f}%",
            f"N {GROUP_B[0]}–{GROUP_B[1]}":      int(log.get('GroupB_N', 0)),
        })

    df_table = pd.DataFrame(rows).set_index("Epoch")

    print("\n" + "━"*75)
    print(f"  📋 BẢNG SO SÁNH 2 NHÓM SỰ KIỆN (cập nhật đến Epoch {epoch})")
    print("━"*75)
    print(df_table.to_string())
    print("━"*75)

    # ★ In thêm delta (chênh lệch) giữa 2 nhóm cho epoch hiện tại
    last = history_logs[-1]
    delta_tau   = last.get('GroupA_Tau',   0) - last.get('GroupB_Tau',   0)
    delta_exact = last.get('GroupA_Exact', 0) - last.get('GroupB_Exact', 0)
    print(f"  Δ Tau  (A–B) epoch {epoch}: {delta_tau:+.4f}  "
          f"| Δ Exact (A–B): {delta_exact:+.2f}%")
    print("━"*75 + "\n")



# =======================================================
# ★ HÀM LƯU / NẠP RUNTIME STATE (GIẢI QUYẾT VẤN ĐỀ 1)
#   Lưu toàn bộ dữ liệu cần thiết để resume liền mạch
# =======================================================
def save_runtime_state(epoch, best_val_tau, patience_counter,
                       best_true_ranks, best_pred_ranks,
                       best_sample_timeline):
    """Lưu state runtime ra JSON — đọc được khi kernel reset."""
    state = {
        "epoch":             epoch,
        "best_val_tau":      best_val_tau,
        "patience_counter":  patience_counter,
        "best_true_ranks":   best_true_ranks,
        "best_pred_ranks":   best_pred_ranks,
        # sample_timeline: (name, events, t_ranks, p_ranks)
        "best_sample_timeline": list(best_sample_timeline) if best_sample_timeline else None,
    }
    with open(RUNTIME_STATE_FILE, 'w', encoding='utf-8') as f:
        json.dump(state, f, ensure_ascii=False)


def load_runtime_state():
    # Ưu tiên file trong working dir (resume lần 2+),
    # fallback về kaggle input (resume lần đầu từ dataset)
    for path in [RUNTIME_STATE_FILE, RUNTIME_STATE_READ]:
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                state = json.load(f)
            if state.get("best_sample_timeline"):
                tl = state["best_sample_timeline"]
                state["best_sample_timeline"] = (tl[0], tl[1], tl[2], tl[3])
            print(f"  ✔ Nạp runtime state từ: {path}")
            return state
    return None


# =======================================================
# 3. KHỞI TẠO MODEL + COMPONENTS
# =======================================================
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

train_loader = DataLoader(
    RankingEventDataset(train_subset, tokenizer),
    batch_size=16, shuffle=True,
    pin_memory=True, num_workers=2  # ← nguy hiểm trong môi trường notebook
)
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base-v2", num_labels=1).to(device)
for module in model.roberta.encoder.layer[:6]:
    for p in module.parameters():
        p.requires_grad = False

optimizer      = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=3e-5)
margin_loss_fn = nn.MarginRankingLoss(margin=1.0)
scheduler      = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2, min_lr=1e-6)


# ★ SỬA 3: BLOCK QUYẾT ĐỊNH RESUME — đọc từ INPUT,
#   ghi ra OUTPUT
# Thay thế block "4. QUYẾT ĐỊNH" cũ
# =======================================================

# Kiểm tra theo thứ tự: working dir trước, input dir sau
def _find_readable(working_path, input_path):
    """Trả về path đọc được, ưu tiên working dir."""
    if os.path.exists(working_path): return working_path
    if os.path.exists(input_path):   return input_path
    return None

ckpt_read    = _find_readable(CHECKPOINT_FILE,    CHECKPOINT_FILE_READ)
runtime_read = _find_readable(RUNTIME_STATE_FILE, RUNTIME_STATE_READ)
metrics_read = _find_readable(EVAL_METRICS_FILE,  EVAL_METRICS_READ)
preds_read   = _find_readable(EVAL_PREDICTIONS_FILE, EVAL_PREDICTIONS_READ)

has_checkpoint    = ckpt_read    is not None
has_runtime_state = runtime_read is not None

if has_checkpoint and has_runtime_state:
    print(f"♻️  Phát hiện checkpoint. Đang RESUME...")
    ckpt = torch.load(ckpt_read, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])

    rs               = load_runtime_state()
    start_epoch      = rs['epoch'] + 1
    best_val_tau     = rs['best_val_tau']
    patience_counter = rs['patience_counter']
    best_true_ranks  = rs['best_true_ranks']
    best_pred_ranks  = rs['best_pred_ranks']
    best_sample_timeline = rs['best_sample_timeline']

    history_logs = pd.read_csv(metrics_read).to_dict('records') \
                   if metrics_read else []
    best_detailed_results = pd.read_csv(preds_read).to_dict('records') \
                            if preds_read else []

    # ★ Copy file input → working dir để các lần ghi sau dùng đường dẫn OUTPUT
    import shutil
    for src, dst in [
        (ckpt_read,    CHECKPOINT_FILE),
        (runtime_read, RUNTIME_STATE_FILE),
        (metrics_read, EVAL_METRICS_FILE),
        (preds_read,   EVAL_PREDICTIONS_FILE),
    ]:
        if src and src != dst and os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"  ✔ Copied {os.path.basename(src)} → {OUTPUT_DIR}/")

    print(f"  ✔ Epoch đã chạy    : {[l['Epoch'] for l in history_logs]}")
    print(f"  ✔ Tiếp tục epoch   : {start_epoch + 1}/{EPOCHS}")
    print(f"  ✔ Best Val Tau     : {best_val_tau:.4f}")
    print(f"  ✔ Patience counter : {patience_counter}/{PATIENCE}")
    print(f"  ✔ Dữ liệu biểu đồ : {len(best_true_ranks)} điểm đã có\n")

elif not has_checkpoint:
    print("🆕 Không có checkpoint. Bắt đầu huấn luyện từ đầu...\n")
    start_epoch, best_val_tau, patience_counter = 0, -1.0, 0
    best_true_ranks, best_pred_ranks = [], []
    best_detailed_results, best_sample_timeline = [], None
    history_logs = []

else:
    print("⚠️  Có checkpoint nhưng thiếu runtime_state. Load weights, reset runtime.\n")
    import shutil
    if ckpt_read != CHECKPOINT_FILE:
        shutil.copy2(ckpt_read, CHECKPOINT_FILE)
    ckpt = torch.load(CHECKPOINT_FILE, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch      = ckpt['epoch'] + 1
    best_val_tau     = ckpt['best_val_tau']
    patience_counter = ckpt['patience_counter']
    best_true_ranks, best_pred_ranks, best_detailed_results = [], [], []
    best_sample_timeline = None
    if preds_read:
        prev = pd.read_csv(preds_read)
        best_true_ranks    = prev['Thực tế'].tolist()
        best_pred_ranks    = prev['Dự đoán'].tolist()
        best_detailed_results = prev.to_dict('records')
    history_logs = pd.read_csv(metrics_read).to_dict('records') if metrics_read else []
    print(f"  ✔ Tiếp tục từ epoch {start_epoch + 1}/{EPOCHS}\n")
print("🚀 BẮT ĐẦU HUẤN LUYỆN...")
for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", mininterval=10.0)

    for batch in loop:
        optimizer.zero_grad()
        pos = model(input_ids=batch['pos_input_ids'].to(device),
                    attention_mask=batch['pos_attention_mask'].to(device)).logits
        neg = model(input_ids=batch['neg_input_ids'].to(device),
                    attention_mask=batch['neg_attention_mask'].to(device)).logits
        loss = margin_loss_fn(pos, neg, torch.ones(pos.size(), device=device))
        loss.backward(); optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # ★ Unpack thêm 6 giá trị nhóm A/B
    (val_tau, val_rho, exact_rate,
     ga_tau, ga_exact, ga_n,
     gb_tau, gb_exact, gb_n,
     t_ranks, p_ranks, details, sample_tl) = validate_model(
        model, df_test, tokenizer, device, VAL_MAX_ROWS, epoch + 1)

    # ★ In kết quả epoch gọn + bảng nhóm
    print(f"\n📊 EPOCH {epoch+1} | Loss: {avg_loss:.4f} | "
          f"Tau: {val_tau:.4f} | Rho: {val_rho:.4f} | Exact: {exact_rate:.2f}%")

    # Ghi log — thêm cột nhóm A/B
    history_logs.append({
        "Epoch":           epoch + 1,
        "Train_Loss":      avg_loss,
        "Val_Tau":         val_tau,
        "Val_Spearman":    val_rho,
        "Exact_Match_Pct": exact_rate,
        # ★ Nhóm A
        "GroupA_Tau":      ga_tau,
        "GroupA_Exact":    ga_exact,
        "GroupA_N":        ga_n,
        # ★ Nhóm B
        "GroupB_Tau":      gb_tau,
        "GroupB_Exact":    gb_exact,
        "GroupB_N":        gb_n,
    })
    pd.DataFrame(history_logs).to_csv(EVAL_METRICS_FILE, index=False)

    # ★ In bảng so sánh 2 nhóm
    print_group_comparison_table(epoch + 1, history_logs)

    if val_tau > best_val_tau:
        best_val_tau, patience_counter = val_tau, 0
        best_true_ranks, best_pred_ranks  = t_ranks, p_ranks
        best_detailed_results             = details
        best_sample_timeline              = sample_tl
        model.save_pretrained(SAVE_PATH)
        tokenizer.save_pretrained(SAVE_PATH)
        pd.DataFrame(best_detailed_results).to_csv(
            EVAL_PREDICTIONS_FILE, index=False, encoding='utf-8-sig')
        print("  🏆 KỶ LỤC MỚI! Đã lưu model + predictions.")
    else:
        patience_counter += 1
        print(f"  ⚠️  Không cải thiện. Patience: {patience_counter}/{PATIENCE}")

    torch.save({
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_tau':         best_val_tau,
        'patience_counter':     patience_counter,
    }, CHECKPOINT_FILE)

    save_runtime_state(epoch, best_val_tau, patience_counter,
                       best_true_ranks, best_pred_ranks,
                       best_sample_timeline)

    scheduler.step(val_tau)

    if patience_counter >= PATIENCE:
        print("🛑 Early stopping.")
        break

    gc.collect(); torch.cuda.empty_cache()


# =======================================================
# 6. XUẤT 6 BIỂU ĐỒ
# =======================================================
print("\n📊 ĐANG TẠO 6 BIỂU ĐỒ PHÂN TÍCH...")
os.system("zip -q -r best_model.zip best_model/")
hist_df = pd.read_csv(EVAL_METRICS_FILE)

# 1. Learning Curve
fig, ax1 = plt.subplots(figsize=(8, 6))
ax1.plot(hist_df['Epoch'], hist_df['Train_Loss'], 'r-o', linewidth=2, label='Train Loss')
ax1.set_ylabel('Loss', color='r'); ax1.set_xlabel('Epoch')
ax2 = ax1.twinx()
ax2.plot(hist_df['Epoch'], hist_df['Val_Tau'], 'b-o', linewidth=2, label='Val Kendall Tau')
ax2.set_ylabel('Kendall Tau', color='b')
plt.title('1. Learning Curve', fontweight='bold')
plt.savefig('plot_1_learning_curve.png', dpi=300, bbox_inches='tight'); plt.close()

# 2. Scatter
plt.figure(figsize=(8, 6))
jt = np.array(best_true_ranks) + np.random.normal(0, 0.15, len(best_true_ranks))
jp = np.array(best_pred_ranks) + np.random.normal(0, 0.15, len(best_pred_ranks))
plt.scatter(jt, jp, alpha=0.3, s=20, color='teal')
plt.plot([1, 15], [1, 15], 'r--', linewidth=2)
plt.xlabel('Thực tế'); plt.ylabel('Dự đoán')
plt.title('2. Scatter Plot (Jitter)', fontweight='bold')
plt.savefig('plot_2_scatter.png', dpi=300, bbox_inches='tight'); plt.close()

# 3. Error Distribution
plt.figure(figsize=(8, 6))
errors = np.array(best_pred_ranks) - np.array(best_true_ranks)
sns.histplot(errors, bins=range(-10, 12), kde=True, color='purple')
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Độ lệch (Pred - True)'); plt.ylabel('Tần suất')
plt.title('3. Error Distribution', fontweight='bold')
plt.savefig('plot_3_error_distribution.png', dpi=300, bbox_inches='tight'); plt.close()

# 4. Rank Density
plt.figure(figsize=(8, 6))
hb = plt.hexbin(best_true_ranks, best_pred_ranks, gridsize=15, cmap='Blues', mincnt=1)
plt.colorbar(hb, label='Số lượng')
plt.plot([1, 15], [1, 15], 'r--', linewidth=2)
plt.xlabel('Thực tế'); plt.ylabel('Dự đoán')
plt.title('4. Rank Correlation Density', fontweight='bold')
plt.savefig('plot_4_rank_density.png', dpi=300, bbox_inches='tight'); plt.close()

# 5. Confusion Matrix
plt.figure(figsize=(8, 6))
max_rank = min(15, max(best_true_ranks) if best_true_ranks else 15)
cm = pd.crosstab(pd.Series(best_true_ranks, name='Thực Tế'),
                 pd.Series(best_pred_ranks,  name='Dự Đoán'))
cm = cm.reindex(index=range(1, max_rank+1), columns=range(1, max_rank+1), fill_value=0)
sns.heatmap(cm, cmap='YlGnBu', annot=False, cbar=True)
plt.title('5. Confusion Matrix', fontweight='bold')
plt.savefig('plot_5_confusion_matrix.png', dpi=300, bbox_inches='tight'); plt.close()

# 6. Timeline Bump Chart
plt.figure(figsize=(8, 6))
if best_sample_timeline:
    cname, evts, t_r, p_r = best_sample_timeline
    for i in range(len(evts)):
        plt.plot([0, 1], [t_r[i], p_r[i]], 'gray', alpha=0.5)
        plt.plot(0, t_r[i], 'bo', markersize=10)
        plt.plot(1, p_r[i], 'ro', markersize=10)
        label = evts[i][:40] + "..." if len(evts[i]) > 40 else evts[i]
        plt.text(-0.05, t_r[i], label, ha='right', va='center', fontsize=9)
    plt.xlim(-0.8, 1.2)
    plt.xticks([0, 1], ['Thực tế', 'Dự đoán'], fontweight='bold')
    plt.gca().invert_yaxis(); plt.axis('off')
    plt.title(f'6. Timeline: {cname}', fontweight='bold')
plt.savefig('plot_6_timeline.png', dpi=300, bbox_inches='tight'); plt.close()


# =======================================================
# [THÊM MỚI] BẢNG TỔNG KẾT CUỐI CÙNG
# =======================================================
print("\n" + "="*75)
print("  📋 TỔNG KẾT TOÀN BỘ QUÁ TRÌNH HUẤN LUYỆN")
print("="*75)

hist_df = pd.read_csv(EVAL_METRICS_FILE)

# Bảng tổng hợp epoch tốt nhất
best_row = hist_df.loc[hist_df['Val_Tau'].idxmax()]
summary = pd.DataFrame([
    {
        "Chỉ số":  "Kendall Tau",
        "Tổng":    f"{best_row['Val_Tau']:.4f}",
        f"{GROUP_A[0]}–{GROUP_A[1]} sự kiện": f"{best_row['GroupA_Tau']:.4f}",
        f"{GROUP_B[0]}–{GROUP_B[1]} sự kiện": f"{best_row['GroupB_Tau']:.4f}",
        "Epoch tốt nhất": int(best_row['Epoch']),
    },
    {
        "Chỉ số":  "Exact Match %",
        "Tổng":    f"{best_row['Exact_Match_Pct']:.2f}%",
        f"{GROUP_A[0]}–{GROUP_A[1]} sự kiện": f"{best_row['GroupA_Exact']:.2f}%",
        f"{GROUP_B[0]}–{GROUP_B[1]} sự kiện": f"{best_row['GroupB_Exact']:.2f}%",
        "Epoch tốt nhất": int(best_row['Epoch']),
    },
    {
        "Chỉ số":  "Số nhân vật (N)",
        "Tổng":    "—",
        f"{GROUP_A[0]}–{GROUP_A[1]} sự kiện": int(best_row['GroupA_N']),
        f"{GROUP_B[0]}–{GROUP_B[1]} sự kiện": int(best_row['GroupB_N']),
        "Epoch tốt nhất": int(best_row['Epoch']),
    },
]).set_index("Chỉ số")

print(summary.to_string())
print("="*75)

# ★ Xuất bảng tổng kết ra CSV riêng
summary.to_csv("summary_group_comparison.csv", encoding='utf-8-sig')
print("\n✅ summary_group_comparison.csv — Bảng so sánh 2 nhóm (epoch tốt nhất)")

print("\n" + "="*55)
print("✅ evaluation_metrics.csv     — Lịch sử điểm số")
print("✅ evaluation_predictions.csv — Kết quả dự đoán")
print("✅ best_model.zip             — Model tốt nhất")
print("✅ last_checkpoint.pth        — Checkpoint model")
print("✅ runtime_state.json         — State resume đầy đủ")
print("✅ plot_1 → plot_6 .png       — Biểu đồ phân tích")
print("="*55)

📂 ĐANG LOAD DATA...
  ✔ 'train_df' đã có trong RAM (276,376 dòng)
  ✔ 'df_test' đã có trong RAM (11,571 dòng)
  🔵 FULL MODE
  Train: 138,188 | Test characters: 1,169



config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

♻️  Phát hiện checkpoint. Đang RESUME...
  ✔ Nạp runtime state từ: /kaggle/input/datasets/mingchan1/model-input/runtime_state.json
  ✔ Copied last_checkpoint.pth → /kaggle/working/
  ✔ Copied runtime_state.json → /kaggle/working/
  ✔ Copied evaluation_metrics.csv → /kaggle/working/
  ✔ Copied evaluation_predictions.csv → /kaggle/working/
  ✔ Epoch đã chạy    : [1, 2, 3, 4, 5, 6, 7]
  ✔ Tiếp tục epoch   : 8/20
  ✔ Best Val Tau     : 0.8756
  ✔ Patience counter : 4/4
  ✔ Dữ liệu biểu đồ : 11571 điểm đã có

🚀 BẮT ĐẦU HUẤN LUYỆN...


Epoch 8/20 [Train]:   0%|          | 0/8637 [00:00<?, ?it/s]

Validation Epoch 8:   0%|          | 0/1169 [00:00<?, ?it/s]


📊 EPOCH 8 | Loss: 0.0238 | Tau: 0.3537 | Rho: 0.4476 | Exact: 1.03%

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📋 BẢNG SO SÁNH 2 NHÓM SỰ KIỆN (cập nhật đến Epoch 8)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
      Train Loss Tau tổng Tau 5–10 Exact 5–10%  N 5–10 Tau 11–15 Exact 11–15%  N 11–15
Epoch                                                                                 
1         0.3876   0.8298   0.8459      34.26%     610    0.8123        5.37%      559
2         0.2117   0.8612   0.8757      39.84%     610    0.8455        7.69%      559
3         0.1439   0.8756   0.8873      44.43%     610    0.8629        8.59%      559
4         0.1067   0.8730   0.8832      42.46%     610    0.8619        5.72%      559
5         0.0796   0.8559   0.8671      34.92%     610    0.8436        1.79%      559
6         0.0610   0.8598   0.8673      35.74%     610    0.8517        3.40%      559
7         0.0356   0.8685  

In [4]:
!ls -F /kaggle/working/

best_model/		    plot_2_scatter.png
best_model.zip		    plot_3_error_distribution.png
data_processed.csv	    plot_4_rank_density.png
evaluation_metrics.csv	    plot_5_confusion_matrix.png
evaluation_predictions.csv  plot_6_timeline.png
last_checkpoint.pth	    runtime_state.json
__notebook__.ipynb	    summary_group_comparison.csv
plot_1_learning_curve.png
